# 03 Volumes

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Store and govern <b>files</b> (CSV, JSON, images, PDFs, wheels) in Unity Catalog <b>volumes</b>: create managed volumes, write and read files with Spark, Python, SQL and <code>dbutils</code>, understand the <code>/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/</code> path, and know how volumes compare with DBFS and direct cloud paths.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Raw data usually arrives as files before it becomes a table. Volumes give those files the same governance as tables (grants, lineage, discovery), replacing the old, ungoverned DBFS root and mount points. In Free Edition, volumes are <b>the</b> place for files, and ingestion with Auto Loader and <code>COPY INTO</code> reads from them.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Volumes in Unity Catalog</b>
<ul><li>A <b>volume</b> is a schema-level object that represents a directory of files: <code>catalog.schema.volume</code></li><li>Path: **<code>/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/&lt;path&gt;</code>**, usable from Spark, SQL, Python (<code>open</code>, <code>os</code>, pandas), shell and <code>dbutils.fs</code></li><li><b>Managed volume</b>: storage chosen by Unity Catalog. Dropping it deletes the files</li><li><b>External volume</b>: points to a path in an external location. Dropping it leaves the files</li><li>Privileges: <code>READ VOLUME</code>, <code>WRITE VOLUME</code> (plus <code>USE CATALOG</code> and <code>USE SCHEMA</code>)</li><li>Volumes hold <b>non-tabular or raw files</b>. Tables hold tabular data, and are registered separately</li></ul>
| Storage option | Governance | Status |<br>|---|---|---|<br>| Unity Catalog volume (<code>/Volumes/...</code>) | Grants, lineage, audit | ✅ Recommended |<br>| Cloud path via external location (<code>abfss://...</code>) | Grants on the location | ✅ For direct cloud access |<br>| DBFS root (<code>dbfs:/FileStore</code>, <code>/tmp</code>) | None, workspace-wide | ⚠️ Legacy, disabled in Free Edition |<br>| Mount points (<code>/mnt/...</code>) | None, shared credentials | ❌ Deprecated |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A logistics company received daily CSV manifests from carriers by SFTP, landed them in <code>/mnt/raw</code> with a storage key shared by everyone, and nobody knew who read what. They moved to an external volume <code>prod.landing.carrier_files</code>: the SFTP process writes there, Auto Loader ingests from it, only the ingestion service principal has <code>WRITE VOLUME</code>, and analysts have <code>READ VOLUME</code> for debugging. Audit logs now show every access.
</div>

## Syntax

```sql
CREATE VOLUME IF NOT EXISTS workspace.governance.files COMMENT 'raw files';
LIST '/Volumes/workspace/governance/files/';
SELECT * FROM read_files('/Volumes/workspace/governance/files/orders/', format => 'csv', header => true);
DROP VOLUME workspace.governance.files;
```

```python
dbutils.fs.ls("/Volumes/workspace/governance/files/")
spark.read.csv("/Volumes/workspace/governance/files/orders/", header=True)
open("/Volumes/workspace/governance/files/notes.txt").read()
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a managed volume <code>files</code> in the <code>governance</code> schema and sets <code>BASE</code> to its path.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.files COMMENT 'Raw files for 06_unity_catalog lessons'")
BASE = f"/Volumes/{CATALOG}/governance/files"

## 1. Write files with plain Python

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a small CSV file and a JSON file into the volume with Python's <code>open</code>, the same way you'd write to a local disk.<br><br>
<b>Why it matters</b><br>Volume paths behave like a file system for Python libraries, so tools like pandas, <code>csv</code> and <code>json</code> work without special APIs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two files written, and their sizes.
</div>

In [0]:
import os, json

os.makedirs(f"{BASE}/orders", exist_ok=True)
with open(f"{BASE}/orders/orders_2024_11_01.csv", "w") as f:
    f.write("order_id,customer_id,amount\n1,1,10.50\n2,2,99.00\n3,1,5.25\n")
with open(f"{BASE}/settings.json", "w") as f:
    json.dump({"source": "carrier_x", "delimiter": ","}, f)

for name in ["orders/orders_2024_11_01.csv", "settings.json"]:
    print(f"{name:<32} {os.path.getsize(f'{BASE}/{name}')} bytes")

## 2. Read files with Spark

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the CSV folder from the volume with Spark, and the JSON settings with Python.<br><br>
<b>Why it matters</b><br>Spark reads a whole folder of files as one DataFrame, which is how raw data in volumes becomes Bronze tables.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 orders with typed columns, and the settings dictionary.
</div>

In [0]:
orders = spark.read.option("header", True).schema("order_id INT, customer_id INT, amount DOUBLE").csv(f"{BASE}/orders/")
orders.show()
print(json.load(open(f"{BASE}/settings.json")))

## 3. SQL access: `LIST` and `read_files`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the volume's files and queries the CSV folder directly with <code>read_files</code>.<br><br>
<b>Why it matters</b><br>SQL users (and SQL warehouses) can explore and query raw files without any Python, and <code>read_files</code> infers the schema.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Works in Free Edition on serverless and SQL warehouses.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The files in the volume, and the 3 orders.
</div>

In [0]:
display(spark.sql(f"LIST '{BASE}/'"))
display(spark.sql(f"SELECT * FROM read_files('{BASE}/orders/', format => 'csv', header => true)"))

## 4. `dbutils.fs` on volumes

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Copies, lists and removes files with <code>dbutils.fs</code>.<br><br>
<b>Why it matters</b><br><code>dbutils.fs</code> is the usual tool for file housekeeping in notebooks and jobs: archiving processed files, cleaning up test data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The copy appears in the archive folder and is then removed.
</div>

In [0]:
dbutils.fs.mkdirs(f"{BASE}/archive")
dbutils.fs.cp(f"{BASE}/orders/orders_2024_11_01.csv", f"{BASE}/archive/orders_2024_11_01.csv")
print([f.name for f in dbutils.fs.ls(f"{BASE}/archive")])
dbutils.fs.rm(f"{BASE}/archive", recurse=True)
print("archive removed")

## 5. From volume to table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Turns the raw CSV files into a managed Delta table, the first step of a medallion pipeline.<br><br>
<b>Why it matters</b><br>This is the pattern for every ingestion lesson in folder 08: files in a volume → Bronze table. The volume keeps the raw files, and the table holds the typed, queryable data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table <code>orders_bronze</code> with 3 rows and a <code>source_file</code> column.
</div>

In [0]:
from pyspark.sql import functions as F

(spark.read.option("header", True).schema("order_id INT, customer_id INT, amount DOUBLE").csv(f"{BASE}/orders/")
 .withColumn("source_file", F.col("_metadata.file_name"))
 .write.mode("overwrite").saveAsTable(f"{DB}.orders_bronze"))
spark.table(f"{DB}.orders_bronze").show(truncate=False)

## 6. Inspect the volume object

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Describes the volume and lists the volumes in the schema.<br><br>
<b>Why it matters</b><br>Volumes are catalog objects with an owner, a type (managed or external), a storage location and a comment, just like tables.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The volume's type (<code>MANAGED</code>), owner and comment.
</div>

In [0]:
display(spark.sql(f"DESCRIBE VOLUME {DB}.files"))
display(spark.sql(f"SHOW VOLUMES IN {DB}"))

## Under the hood

```
open("/Volumes/workspace/governance/files/x.csv")
  └─ Unity Catalog checks READ VOLUME / WRITE VOLUME (+ USE CATALOG, USE SCHEMA)
       └─ maps /Volumes/<c>/<s>/<v>/ to the volume's cloud location
            └─ short-lived, scoped credentials → read/write in cloud storage
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata</span> the volume is a catalog entry; the files are ordinary objects in cloud storage under the volume's location.

In [0]:
print(sorted(os.listdir(BASE))[:10])

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: writing to <code>dbfs:/FileStore</code> or <code>/tmp</code> in Free Edition</b><br>
The public DBFS root is disabled. Use a volume path.<br><br>
**⛔ Problem: <code>PERMISSION_DENIED</code> reading a volume**<br>You need <code>READ VOLUME</code> on the volume plus <code>USE CATALOG</code> and <code>USE SCHEMA</code>.<br><br>
**⛔ Problem: path typo like <code>/Volume/</code> or a missing schema level**<br>The path is always <code>/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/...</code>.<br><br>
<b>⛔ Problem: storing tabular data only as files</b><br>Files are for raw input. Load them into tables to get schemas, Delta features and SQL access.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is a Unity Catalog volume?</b><br>
A schema-level object that governs a directory of files in cloud storage, accessible at <code>/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/</code>. It brings grants, lineage and auditing to non-tabular and raw files.<br><br>

<b>🎤 2. What is the difference between managed and external volumes?</b><br>
A managed volume's storage is chosen and managed by Unity Catalog, and dropping it deletes the files. An external volume points to a path in an external location, and dropping it leaves the files in place.<br><br>

<b>🎤 3. Why use volumes instead of DBFS or mount points?</b><br>
The DBFS root and mounts aren't governed by Unity Catalog: anyone in the workspace can access them, and mounts share one credential. Volumes have fine-grained privileges (<code>READ VOLUME</code>, <code>WRITE VOLUME</code>), audit logging and lineage.<br><br>

<b>🎤 4. How can you read files in a volume?</b><br>
With Spark readers or Auto Loader, SQL (<code>read_files</code>, <code>LIST</code>, <code>COPY INTO</code>), standard Python file APIs and libraries, and <code>dbutils.fs</code>.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which path format refers to files in a Unity Catalog volume?</b><br>
A. <code>dbfs:/mnt/&lt;volume&gt;/file.csv</code><br>
B. <code>/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/file.csv</code><br>
C. <code>/Workspace/&lt;volume&gt;/file.csv</code><br>
D. <code>hive_metastore.&lt;volume&gt;.file</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which privilege lets a user read files in a volume (assuming they have <code>USE CATALOG</code> and <code>USE SCHEMA</code>)?</b><br>
A. <code>SELECT</code><br>
B. <code>READ VOLUME</code><br>
C. <code>READ FILES</code><br>
D. <code>EXECUTE</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> (<code>READ FILES</code> applies to external locations.)</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write two JSON-lines files of customer records into <code>{BASE}/customers/</code></li><li>Read the folder with Spark, add the source file name, and save it as table <code>customers_bronze</code></li><li>(Databricks) Use <code>LIST</code> to list the folder, then remove it with <code>dbutils.fs.rm</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Two JSON-lines files
os.makedirs(f"{BASE}/customers", exist_ok=True)
for i, rows in enumerate([[{"id": 1, "name": "Asha"}, {"id": 2, "name": "Ben"}], [{"id": 3, "name": "Chen"}]]):
    with open(f"{BASE}/customers/part_{i}.json", "w") as f:
        f.write("\n".join(json.dumps(r) for r in rows))

# 2. Folder -> Bronze table with the source file name
(spark.read.schema("id INT, name STRING").json(f"{BASE}/customers/")
 .withColumn("source_file", F.col("_metadata.file_name"))
 .write.mode("overwrite").saveAsTable(f"{DB}.customers_bronze"))
spark.table(f"{DB}.customers_bronze").orderBy("id").show()

In [0]:
# 3. List and clean up
display(spark.sql(f"LIST '{BASE}/customers/'"))
dbutils.fs.rm(f"{BASE}/customers", recurse=True)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Volumes govern files: <code>catalog.schema.volume</code>, path <code>/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/</code></li><li>Managed volume: UC-managed storage, <code>DROP</code> deletes files. External volume: path in an external location, <code>DROP</code> keeps files</li><li>Works with Spark, SQL (<code>LIST</code>, <code>read_files</code>), Python file APIs and <code>dbutils.fs</code></li><li>Privileges: <code>READ VOLUME</code>, <code>WRITE VOLUME</code></li><li>Prefer volumes over the DBFS root and mounts (disabled or deprecated)</li></ul>
</div>

| Task | Command |
|---|---|
| Create | `CREATE VOLUME c.s.v` |
| List | `LIST '/Volumes/c/s/v/'`, `dbutils.fs.ls(...)` |
| Query files | `read_files('/Volumes/c/s/v/dir/', format => 'csv')` |
| Python | `open('/Volumes/c/s/v/file')` |
| Describe | `DESCRIBE VOLUME c.s.v` |
| Grant | `GRANT READ VOLUME ON VOLUME c.s.v TO ...` |

## Git commit

```
git add 06_unity_catalog/03_volumes.ipynb
git commit -m "add 06_03 volumes notebook"
```